# Library

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load Data

In [3]:
data = pd.read_csv("data/clean_final_data_mitula_lamudi.csv")
data.head(10)

,Harga_Asli,kecamatan,bedroom,bathrooms,jumlah_guest_toilet,list_facilities,list_buildings,lantai,area_digunakan,luas_tanah,latitude,longitude
0,385000000,Wonoayu,2,1,1,"Internet, Air, Tanpa perabotan, Halaman, Listrik","Keamanan, Taman, Keamanan 24 jam, Area anak-an...",1,34,10,-7.437160,112.622220
1,540000000,Sedati,2,1,1,"Garasi, AC, Kabel video, Air, Tanpa perabotan,...","Keamanan, Gym, Taman, Rumah jaga, Keamanan 24 ...",1,32,32,-7.395451,112.769745
2,1960000000,Candi,4,3,3,"Garasi, AC, Dapur lengkap, Tangki air, Listrik...","Area anak-anak, Rumah jaga, Taman, Keamanan 24...",3,72,32,-7.458483,112.700560
3,950000000,Candi,3,2,2,"Garasi, Halaman, Internet, Tanpa perabotan","Area anak-anak, Taman atap, Taman, Rumah jaga",2,72,32,-7.484486,112.725384
4,691000000,Sukodono,2,2,2,"Garasi, Halaman, Tanpa perabotan, Internet, Li...","Keamanan, Rumah jaga, Taman, Keamanan 24 jam",2,72,32,-7.391429,112.698825
5,650000000,Candi,2,1,1,"Listrik, Fully fenced, Tanpa perabotan",Tidak Ada,1,30,35,-7.484927,112.743517
6,390000000,Sedati,2,1,1,"Garasi, Halaman, Listrik, Fully fenced, Pemand...","Keamanan, Area anak-anak, Rumah jaga, Gym, Tam...",1,35,35,-7.400810,112.803280
7,475000000,Candi,2,1,0,Tanpa perabotan,Tidak Ada,1,35,35,-7.458213,112.690480
8,455000000,Sukodono,2,1,1,Tanpa perabotan,Tidak Ada,1,35,35,-7.388464,112.703773
9,355000000,Tulangan,2,1,1,Tanpa perabotan,Tidak Ada,1,35,35,-7.449429,112.639137


# Klasifikasi akses jalan

In [5]:
import pandas as pd
import osmnx as ox
import os
import ast
from tqdm.auto import tqdm

# Mengaktifkan progress bar untuk pandas apply
tqdm.pandas()

# ==========================================
# BAGIAN 1: PERSIAPAN PETA (LOAD GRAPH)
# ==========================================
graph_filename = "sidoarjo_all_network.graphml"
place_name = "Kabupaten Sidoarjo, Jawa Timur, Indonesia"

if os.path.exists(graph_filename):
    print(f"[INFO] File '{graph_filename}' ditemukan. Memuat peta dari local storage...")
    G = ox.load_graphml(graph_filename)
else:
    print(f"[INFO] File '{graph_filename}' tidak ada.")
    print(f"[INFO] Sedang mendownload peta '{place_name}' (network_type='all')...")
    # PENTING: Gunakan 'all' agar jalan setapak/gang (footway) ikut terambil
    G = ox.graph_from_place(place_name, network_type='all')
    ox.save_graphml(G, graph_filename)
    print("[INFO] Download selesai dan file tersimpan.")

print("[INFO] Peta siap digunakan.")

# ==========================================
# BAGIAN 2: FUNGSI UTILITAS (CLEANING & SEARCH)
# ==========================================
def clean_osm_tag(tag_value):
    """Membersihkan tag OSM yang tidak konsisten (list vs string)."""
    if tag_value is None:
        return "unknown"
    if isinstance(tag_value, list):
        return tag_value[0]
    if isinstance(tag_value, str) and tag_value.startswith('[') and tag_value.endswith(']'):
        try:
            actual_list = ast.literal_eval(tag_value)
            if isinstance(actual_list, list) and len(actual_list) > 0:
                return actual_list[0]
        except (ValueError, SyntaxError):
            pass
    return tag_value

def get_road_type(lat, lon, graph):
    """Mencari tipe jalan terdekat dari koordinat."""
    try:
        # Cari edge terdekat
        u, v, key = ox.distance.nearest_edges(graph, lon, lat)
        # Ambil data edge
        edge_data = graph.get_edge_data(u, v, key)
        # Ambil tag highway
        raw_road_type = edge_data.get('highway', 'unknown')
        return clean_osm_tag(raw_road_type)
    except Exception:
        return "unknown"

# ==========================================
# BAGIAN 3: DEFINISI LOGIKA SKORING (MAPPING)
# ==========================================
def get_road_score(road_type):
    """
    Mengubah tipe jalan string menjadi skor ordinal 1-5.
    """
    # Normalisasi ke lowercase untuk keamanan
    if isinstance(road_type, str):
        rt = road_type.lower()
    else:
        return 1 # Default jika unknown/error dianggap akses sulit

    # Logika Bisnis
    if rt in ['motorway', 'trunk', 'primary', 'motorway_link', 'trunk_link', 'primary_link']:
        return 5 # Jalan Protokol / Besar
    elif rt in ['secondary', 'tertiary', 'secondary_link', 'tertiary_link']:
        return 4 # Jalan Raya Penghubung
    elif rt in ['residential', 'living_street']:
        return 3 # Jalan Perumahan (Ideal)
    elif rt in ['service', 'unclassified']:
        return 2 # Jalan Sempit / Satu Mobil
    elif rt in ['footway', 'path', 'track', 'pedestrian']:
        return 1 # Gang Senggol
    else:
        return 1 # Fallback

# ==========================================
# BAGIAN 4: FUNGSI WRAPPER DATAFRAME
# ==========================================
def calculate_access_score(row):
    """Fungsi ini akan diaplikasikan per baris pada DataFrame"""
    try:
        # Pastikan variable G (Graph) diambil dari scope global
        r_type = get_road_type(row['latitude'], row['longitude'], G)
        
        # Mapping ke skor
        score = get_road_score(r_type)
        
        return pd.Series([r_type, score], index=['osm_road_type', 'road_score'])
    except Exception as e:
        return pd.Series(['error', 1], index=['osm_road_type', 'road_score'])

# ==========================================
# BAGIAN 5: EKSEKUSI PADA DATA
# ==========================================
print("\n[PROCESS] Sedang memproses klasifikasi jalan... (Mohon tunggu)")

# Pastikan DataFrame 'data' sudah diload sebelumnya (seperti di screenshot Anda)
if 'data' in locals():
    data[['osm_road_type', 'road_score']] = data.progress_apply(calculate_access_score, axis=1)

    # --- CEK HASIL ---
    print("\n--- Hasil Klasifikasi (5 Baris Teratas) ---")
    display(data[['kecamatan', 'latitude', 'longitude', 'osm_road_type', 'road_score']].head())

    # --- VISUALISASI SEBARAN ---
    print("\n--- Distribusi Tipe Jalan ---")
    print(data['road_score'].value_counts().sort_index(ascending=False))
else:
    print("[ERROR] Variabel 'data' belum ditemukan. Pastikan Anda sudah menjalankan pd.read_csv() sebelumnya.")

[INFO] File 'sidoarjo_all_network.graphml' ditemukan. Memuat peta dari local storage...
[INFO] Peta siap digunakan.

[PROCESS] Sedang memproses klasifikasi jalan... (Mohon tunggu)


  0%|          | 0/1241 [00:00<?, ?it/s]


--- Hasil Klasifikasi (5 Baris Teratas) ---


,kecamatan,latitude,longitude,osm_road_type,road_score
0,Wonoayu,-7.437160,112.622220,primary,5
1,Sedati,-7.395451,112.769745,residential,3
2,Candi,-7.458483,112.700560,tertiary,4
3,Candi,-7.484486,112.725384,residential,3
4,Sukodono,-7.391429,112.698825,residential,3



--- Distribusi Tipe Jalan ---
road_score
5     87
4    244
3    764
2    135
1     11
Name: count, dtype: int64


In [6]:
# Tentukan nama file output
output_filename = "data/data_properti_sidoarjo_with_road_score.csv"

# Simpan ke CSV
# index=False digunakan agar nomor baris (0, 1, 2...) tidak ikut tersimpan sebagai kolom
data.to_csv(output_filename, index=False)

print(f"✅ Data berhasil disimpan ke file: {output_filename}")

# (Opsional) Cek apakah file benar-benar terbentuk
import os
if os.path.exists(output_filename):
    file_size = os.path.getsize(output_filename) / 1024 # Konversi ke KB
    print(f"Ukuran file: {file_size:.2f} KB")

✅ Data berhasil disimpan ke file: data/data_properti_sidoarjo_with_road_score.csv
Ukuran file: 187.50 KB
